# AIxodia daemon (Kaggle CPU)

สร้าง `ai-engine` จากโค้ดใน repo แล้วรันพร้อม Cloudflare quick tunnel
daemon บูตแบบไม่มี secret ของตัวเอง — token และ provider keys มาจากมือถือครั้งแรกที่เชื่อม
เวลานับจากเซลล์ถัดไป ไม่ใช่จากตอน daemon เริ่ม เพราะ clone/Go/test กินเวลาไปแล้ว

เซลล์บริการเปิดหน้าต่าง handover 30 นาทีก่อนถึงเพดาน แล้วยังให้บริการอยู่จนกว่าตัวใหม่
ประกาศ URL ตัวเอง — ตัวใหม่ถูกสั่งจาก `.github/workflows/kaggle-deploy.yml` เพราะ kernel
เรียก kernel อื่นไม่ได้


In [ ]:
import base64, json, os, time, urllib.request, urllib.error

# The kernel's own start, recorded in the first code cell so the budget is
# measured against the whole run. The clone, the Go toolchain and the test suite
# are all spent before the service starts, so counting uptime from the daemon
# would ask for more time than Kaggle allows and the kernel would be cut
# mid-answer.
KERNEL_T0 = time.time()
KERNEL_BUDGET_S = int(os.environ.get("AIX_KERNEL_BUDGET_S", "43200"))  # 12h
HANDOVER_LEAD_S = int(os.environ.get("AIX_HANDOVER_LEAD_S", "1800"))    # 30m before the cap

# The gateway's own entry config. It carries no secret: the Cloudflare token
# arrives in the phone's Authorization header and is held in memory only
# (REQ-046(3), CON-012).
ENTRY = json.loads(base64.b64decode('ewogICJtb2JpbGUiOiB7CiAgICAiZW5hYmxlZCI6IHRydWUsCiAgICAiY2xvdWRmbGFyZV9hcGkiOiAiIiwKICAgICJkMV9kYXRhYmFzZSI6ICJhaXhvZGlhIiwKICAgICJsaXN0ZW4iOiAiMTI3LjAuMC4xOjE4NzkwIiwKICAgICJwdWJsaWNfbGlzdGVuIjogIiIsCiAgICAidHVubmVsIjogdHJ1ZSwKICAgICJjbG91ZGZsYXJlZCI6ICJjbG91ZGZsYXJlZCIsCiAgICAic3luY19jb25maWciOiBmYWxzZSwKICAgICJzeW5jX3Nlc3Npb25zIjogdHJ1ZQogIH0KfQ==').decode())

# Only used to read and write the handover announcement in D1. Without it the
# notebook says so and simply stops at its cap, which leaves a gap rather than a
# handover.
CF_TOKEN = os.environ.get("CF_TOKEN", "")

print("kernel start recorded budget=%ds handover_lead=%ds listen=%s tunnel=%s token_len=%d" % (
    KERNEL_BUDGET_S, HANDOVER_LEAD_S, ENTRY["mobile"]["listen"],
    ENTRY["mobile"]["tunnel"], len(CF_TOKEN)))



# The two D1 helpers the handover needs. This notebook has no access to the
# mirror's, so it carries its own: the announcement is a single row read and
# written over the same REST endpoint the daemon already uses (REQ-046(4)).
def _api_base():
    return "https://api.cloudflare.com/client/v4"


def _cf_headers(token):
    return {"Authorization": "Bearer " + token, "Content-Type": "application/json"}


def _cf_get(path, token):
    req = urllib.request.Request(_api_base() + path, headers=_cf_headers(token))
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return json.load(r)
    except urllib.error.HTTPError as e:
        try:
            return json.loads(e.read().decode())
        except Exception:
            return {"success": False, "errors": [{"code": e.code, "message": "http"}]}


def pick_target(token):
    """The account and database are discovered from the token, never configured."""
    accs = (_cf_get("/accounts", token).get("result") or [])
    if not accs:
        return None, "accounts ว่าง — token ไม่มีสิทธิ์ Account หรือหมดอายุ"
    acc = accs[0]["id"]
    dbs = (_cf_get(f"/accounts/{acc}/d1/database", token).get("result") or [])
    want = ENTRY["mobile"].get("d1_database") or "aixodia"
    db = next((d for d in dbs if d.get("name") == want), dbs[0] if dbs else None)
    if db is None:
        return None, "ไม่พบ D1 ชื่อ %r (มี: %s)" % (want, [d.get("name") for d in dbs])
    return (acc, db.get("uuid") or db.get("id")), None


def d1_query(acc, dbid, token, sql, params=()):
    body = json.dumps({"sql": sql, "params": list(params)}).encode()
    req = urllib.request.Request(
        f"{_api_base()}/accounts/{acc}/d1/database/{dbid}/query",
        data=body, headers=_cf_headers(token))
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            env = json.load(r)
    except urllib.error.HTTPError as e:
        try:
            env = json.loads(e.read().decode())
        except Exception:
            env = {"success": False}
    if not env.get("success") or not all(s.get("success", True) for s in (env.get("result") or [])):
        raise RuntimeError("D1 query ล้มเหลว: %r" % (env.get("errors"),))
    return env



## Phase A — tests (CPU)

clone `Tulipskun/ai` (public) ลง `/tmp/aixodia`, ติดตั้ง Go ตาม `go.mod`, แล้วรัน `go test ./...`
ถ้า test แดง kernel จะหยุดตรงนี้ ไม่ไป Phase B

In [ ]:
import json, os, re, shutil, subprocess, tarfile, urllib.request
from datetime import datetime, timezone

def emit(event, **fields):
    rec = {"t": datetime.now(timezone.utc).isoformat(), "event": event}
    rec.update(fields)
    print(json.dumps(rec, ensure_ascii=False), flush=True)

WORK = "/tmp/aixodia"
REPO = os.path.join(WORK, "ai")
os.makedirs(WORK, exist_ok=True)

def sh(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)

# 1. clone public repo (no auth needed)
if not os.path.isdir(os.path.join(REPO, ".git")):
    emit("clone_start", repo="Tulipskun/ai-engine")
    r = sh(["git", "clone", "--depth", "1", "https://github.com/Tulipskun/ai-engine", REPO])
    print((r.stdout + r.stderr)[-2000:], flush=True)
    if r.returncode != 0:
        emit("clone_failed", exit=r.returncode)
        raise SystemExit(3)
else:
    sh(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"])
    sh(["git", "-C", REPO, "reset", "--hard", "origin/main"])
sha = sh(["git", "-C", REPO, "rev-parse", "--short", "HEAD"]).stdout.strip()
emit("repo_ready", sha=sha)

# 2. go toolchain (version from go.mod)
env = dict(os.environ)
env["GOCACHE"] = os.path.join(WORK, "gocache")
env["GOPATH"] = os.path.join(WORK, "gopath")
env["GOFLAGS"] = "-count=1"
go_bin = shutil.which("go")
r = sh(["go", "version"]) if go_bin else subprocess.CompletedProcess(["go"], 1, "", "go not in PATH")
if r.returncode != 0:
    m = re.search(r"^go (\S+)", open(os.path.join(REPO, "go.mod")).read(), re.M)
    gover = m.group(1) if m else "1.25.0"
    emit("go_install_start", version=gover)
    tgz = os.path.join(WORK, "go.tgz")
    urllib.request.urlretrieve(f"https://go.dev/dl/go{gover}.linux-amd64.tar.gz", tgz)
    with tarfile.open(tgz) as tf:
        tf.extractall(os.path.join(WORK, "gotools"))
    os.remove(tgz)
    env["PATH"] = os.path.join(WORK, "gotools", "go", "bin") + ":" + env["PATH"]
    r = sh(["go", "version"], env=env)
print("go:", (r.stdout or r.stderr).strip(), flush=True)
if r.returncode != 0:
    emit("go_missing")
    raise SystemExit(3)
globals()["GO_ENV"] = env

# 3. tests (fail here -> kernel stops, no service)
emit("test_start", timeout="25m")
p = subprocess.Popen(["go", "test", "-timeout", "25m", "./..."],
                     cwd=REPO, env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
tail = []
for line in p.stdout:
    s = line.rstrip()
    tail.append(s)
    if len(tail) > 200:
        tail.pop(0)
    if s.startswith(("ok ", "FAIL", "---", "panic")):
        print(s, flush=True)
rc = p.wait()
emit("test_done", ok=(rc == 0), exit=rc, tail=tail[-40:])
print(f"TEST_EXIT={rc}", flush=True)
if rc != 0:
    raise SystemExit(3)


## Phase B — daemon service (CPU, สูงสุด 12 ชม.)

build `ai-engine` จากโค้ดที่ผ่าน Phase A แล้วรันพร้อม quick tunnel
daemon บูตแบบไม่มี secret: token/keys มาจากมือถือครั้งแรกที่เชื่อม (D1 hydrate)
เวลาที่ใช้นับจากเซลล์แรก ไม่ใช่จากตอน daemon เริ่ม เพราะ clone/Go/test กินเวลาไปแล้ว
ก่อนถึงเพดาน 30 นาที cell นี้จะเปิดหน้าต่าง handover และยังให้บริการอยู่ รอตัวใหม่ประกาศ URL
ตัวใหม่ถูกสั่งจากข้างนอกด้วย `.github/workflows/kaggle-deploy.yml` เพราะ kernel เรียก kernel อื่นไม่ได้

cell นี้ดัก `TUNNEL_URL` จาก log แล้วพิมพ์ซ้ำให้หาง่าย — เอา URL นี้ไปตั้งในแอป


In [ ]:
import json, os, re, shutil, stat, subprocess, threading, time, urllib.request
from datetime import datetime, timezone

def emit(event, **fields):
    rec = {"t": datetime.now(timezone.utc).isoformat(), "event": event}
    rec.update(fields)
    print(json.dumps(rec, ensure_ascii=False), flush=True)

WORK = "/tmp/aixodia"
REPO = os.path.join(WORK, "ai")
env = dict(globals().get("GO_ENV", os.environ))

# Measured against the kernel, not against the daemon. Cloning, installing Go
# and running the tests all happen first, so a cap counted from here would ask
# for more time than the kernel has and it would be killed mid-answer.
KERNEL_T0 = globals().get("KERNEL_T0", time.time())
KERNEL_BUDGET_S = int(os.environ.get("AIX_KERNEL_BUDGET_S", "43200"))  # 12h
HANDOVER_LEAD_S = int(os.environ.get("AIX_HANDOVER_LEAD_S", "1800"))    # start handing over 30m early
HARD_STOP_MARGIN_S = int(os.environ.get("AIX_HARD_STOP_MARGIN_S", "120"))

hard_deadline = KERNEL_T0 + KERNEL_BUDGET_S
handover_at = hard_deadline - HANDOVER_LEAD_S
stop_at = hard_deadline - HARD_STOP_MARGIN_S
emit("budget", budget_s=KERNEL_BUDGET_S, handover_at_s=int(handover_at - time.time()),
     stop_at_s=int(stop_at - time.time()),
     note="measured from the first cell, not from the service")

# 1. state + entry.json (ENTRY from cell 1, carries no secret)
STATE = os.path.join(os.path.expanduser("~"), ".local", "share", "ai")
os.makedirs(os.path.join(STATE, "config"), exist_ok=True)
mobile = dict(ENTRY["mobile"])
# Fresh boot has no local config: hydrate provider/system catalogue from D1
# on the first verified handshake (REQ-046(4)). PushConfig skips missing
# files, so an empty state can never clobber D1 on shutdown.
mobile["sync_config"] = True
with open(os.path.join(STATE, "config", "entry.json"), "w") as f:
    json.dump({"mobile": mobile}, f, indent=2)
emit("entry_written", listen=ENTRY["mobile"]["listen"], tunnel=ENTRY["mobile"]["tunnel"])

# 2. build daemon
DAEMON = os.path.join(WORK, "ai-engine")
# -ldflags stamps the build sha into the binary, and the daemon writes it
# into the D1 nodes row, so a D1 query names the build that is serving.
SHA = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"],
                    capture_output=True, text=True).stdout.strip() or "unknown"
r = subprocess.run(["go", "build", "-ldflags", "-X main.version=" + SHA,
                    "-o", DAEMON, "./cmd/ai-engine"], cwd=REPO,
                   env=env, capture_output=True, text=True)
print((r.stdout + r.stderr)[-3000:], flush=True)
if r.returncode != 0 or not os.path.isfile(DAEMON):
    emit("build_failed", exit=r.returncode)
    raise SystemExit(4)
emit("build_ok", path=DAEMON)

# 3. cloudflared binary (daemon spawns `cloudflared` from PATH)
BINDIR = os.path.join(WORK, "bin")
os.makedirs(BINDIR, exist_ok=True)
cf = shutil.which("cloudflared") or os.path.join(BINDIR, "cloudflared")
if not (shutil.which("cloudflared") or os.path.isfile(cf)):
    emit("cloudflared_download_start")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    os.chmod(cf, os.stat(cf).st_mode | stat.S_IEXEC)
env["PATH"] = BINDIR + ":" + env["PATH"]
print("cloudflared:", subprocess.run(
    ["cloudflared", "--version"], capture_output=True, text=True, env=env
).stdout.strip()[:120], flush=True)

# 4. run
emit("service_start", budget_s=KERNEL_BUDGET_S,
     note="set the app endpoint to TUNNEL_URL below; phone owns keys")
proc = subprocess.Popen([DAEMON], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=env)
url_re = re.compile(r"https://[A-Za-z0-9.-]+\.trycloudflare\.com")
seen, last, last_hb = [], [], 0
def reader():
    for line in proc.stdout:
        s = line.rstrip()
        last.append(s)
        if len(last) > 200:
            last.pop(0)
        print(s, flush=True)
        m = url_re.search(s)
        if m and m.group(0) not in seen:
            seen.append(m.group(0))
            print("\n===== TUNNEL_URL=%s =====\n" % m.group(0), flush=True)
            emit("tunnel_url", url=m.group(0))
threading.Thread(target=reader, daemon=True).start()

# 5. handover
#
# A Kaggle kernel cannot start another Kaggle kernel: there is no call from
# inside a running one, and the token that would allow it must not exist here
# (CON-012). So the successor is started from outside — `kaggle kernels push`
# runs a kernel, which is what .github/workflows/kaggle-deploy.yml does on a
# schedule. What this side of the handover owns is the part that has to happen
# here: stay serving until the successor is actually up, so there is no gap, and
# stop only once it has announced itself.
#
# Announcements go in the state table rather than the nodes row. The nodes row is
# the single address the phone reads, so two overlapping instances writing it
# would make the address flip between them; the state key is per instance and is
# only read by the other instance looking for a successor.
STARTED_AT = int(time.time())
INSTANCE = "%s-%d" % (SHA, STARTED_AT)
CF_TOKEN = os.environ.get("CF_TOKEN", "")
_d1 = globals().get("d1_query")
_pick = globals().get("pick_target")
_handover_target = None
if CF_TOKEN and _d1 and _pick:
    try:
        _handover_target, _err = _pick(CF_TOKEN)
    except Exception as e:
        emit("handover_target_failed", error=str(e)[:160])
else:
    emit("handover_disabled", reason="ไม่มี CF_TOKEN — จะหยุดตามเวลาตามปกติ")

def d1_read_state(key):
    try:
        env_json = _d1(*_handover_target, CF_TOKEN, "SELECT value FROM state WHERE key=? LIMIT 1", (key,))
        rows = (((env_json or {}).get("result") or [])[0] or {}).get("results") or []
        if rows and rows[0].get("value"):
            return json.loads(rows[0]["value"])
    except Exception:
        return None
    return None

def announce_ready(url):
    if not _handover_target:
        return
    payload = json.dumps({"instance": INSTANCE, "sha": SHA, "tunnel": url,
                          "at": STARTED_AT}, ensure_ascii=False)
    try:
        _d1(*_handover_target, CF_TOKEN,
            "INSERT INTO state(key,value,updated_at) VALUES(?,?,unixepoch()) "
            "ON CONFLICT(key) DO UPDATE SET value=excluded.value,updated_at=unixepoch()",
            ("handover/ready", payload))
        emit("handover_announced", instance=INSTANCE, tunnel=url)
    except Exception as e:
        emit("handover_announce_failed", error=str(e)[:160])

def successor_seen():
    rec = d1_read_state("handover/ready")
    if not rec or rec.get("instance") == INSTANCE:
        return None
    # Only a record written after this instance started counts. The key holds
    # one record at a time, so without this a fresh instance reads the previous
    # one's announcement on its very first poll and stands itself down.
    if int(rec.get("at") or 0) <= STARTED_AT:
        return None
    return rec

announced = False
handover_started = False
last_poll = 0.0
POLL_EVERY_S = int(os.environ.get("AIX_HANDOVER_POLL_S", "60"))
while True:
    rc = proc.poll()
    if rc is not None:
        emit("service_down", exit=rc, tail=last[-30:])
        raise SystemExit(4)
    now = time.time()
    up = now - KERNEL_T0

    if not announced and seen:
        announced = True
        announce_ready(seen[-1])

    # Asking D1 every ten seconds is a few hundred queries an hour for an answer
    # that changes on the scale of hours.
    successor = None
    if now - last_poll >= POLL_EVERY_S:
        last_poll = now
        successor = successor_seen()
    if successor:
        # The replacement is up and has published its address, so this one can go
        # without the phone ever losing the tunnel.
        emit("handover_complete", to=successor.get("instance"), to_sha=successor.get("sha"),
             to_tunnel=successor.get("tunnel"), uptime_s=int(up))
        break

    if not handover_started and now >= handover_at:
        handover_started = True
        emit("handover_window_open", remaining_s=int(hard_deadline - now),
             note="รอตัวใหม่ประกาศตัวเอง; ถ้าไม่มีจะหยุดเองตอน stop_at")

    if now >= stop_at:
        emit("service_done",
             reason=("handover_no_successor" if handover_started else "time_cap"),
             uptime_s=int(up), tunnel=seen[-1] if seen else None)
        break

    if up - last_hb >= 300:
        last_hb = up
        emit("heartbeat", uptime_s=int(up), remaining_s=int(hard_deadline - now),
             tunnel=seen[-1] if seen else None, handing_over=handover_started)
    time.sleep(10)

proc.terminate()
try:
    proc.wait(timeout=20)
except subprocess.TimeoutExpired:
    proc.kill()
emit("service_stopped", uptime_s=int(time.time() - KERNEL_T0), exit=proc.returncode)


